In [2]:
import pandas as pd
import numpy as np

claims = pd.read_csv("../data/raw/Car_Insurance_Claim.csv")
motor  = pd.read_csv("../data/raw/motor_data14-2018.csv")

In [8]:
claims = claims.drop_duplicates()
claims = claims.dropna()

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

cat_cols = ["AGE", "GENDER", "RACE", "DRIVING_EXPERIENCE",
            "EDUCATION", "INCOME", "VEHICLE_YEAR", "VEHICLE_TYPE"]

claims_encoded = claims.copy()
for col in cat_cols:
    claims_encoded[col] = le.fit_transform(claims_encoded[col].astype(str))

print("Claims after cleaning:", claims_encoded.shape)
claims_encoded.head()

Claims after cleaning: (8149, 19)


,ID,AGE,GENDER,RACE,DRIVING_EXPERIENCE,EDUCATION,INCOME,CREDIT_SCORE,VEHICLE_OWNERSHIP,VEHICLE_YEAR,MARRIED,CHILDREN,POSTAL_CODE,ANNUAL_MILEAGE,VEHICLE_TYPE,SPEEDING_VIOLATIONS,DUIS,PAST_ACCIDENTS,OUTCOME
0,569520,3,0,0,0,0,2,0.629027,1.0,0,0.0,1.0,10238,12000.0,0,0,0,0,0.0
1,750365,0,1,0,0,1,1,0.357757,0.0,1,0.0,0.0,10238,16000.0,0,0,0,0,1.0
2,199901,0,0,0,0,0,3,0.493146,1.0,1,0.0,0.0,10238,11000.0,0,0,0,0,0.0
3,478866,0,1,0,0,2,3,0.206013,1.0,1,0.0,1.0,32765,11000.0,0,0,0,0,0.0
4,731664,1,1,0,1,1,3,0.388366,1.0,1,0.0,0.0,32765,12000.0,0,2,0,1,1.0


In [10]:
claims_encoded["risk_score"] = (
    claims_encoded["PAST_ACCIDENTS"] * 2 +
    claims_encoded["SPEEDING_VIOLATIONS"] +
    claims_encoded["DUIS"] * 3
)

claims_encoded["high_mileage_flag"] = (claims_encoded["ANNUAL_MILEAGE"] > 15000).astype(int)

claims_encoded["risk_tier"] = pd.cut(
    claims_encoded["risk_score"],
    bins=[-1, 2, 5, 100],
    labels=["Low", "Medium", "High"]
)

In [11]:
motor = motor.drop_duplicates()
motor["CLAIM_PAID"]     = pd.to_numeric(motor["CLAIM_PAID"],     errors="coerce").fillna(0)
motor["INSURED_VALUE"]  = pd.to_numeric(motor["INSURED_VALUE"],  errors="coerce")
motor["PREMIUM"]        = pd.to_numeric(motor["PREMIUM"],        errors="coerce")
motor["PROD_YEAR"]      = pd.to_numeric(motor["PROD_YEAR"],      errors="coerce")
motor = motor.dropna(subset=["INSURED_VALUE", "PREMIUM"])

motor["had_claim"]      = (motor["CLAIM_PAID"] > 0).astype(int)
motor["premium_rate"]   = motor["PREMIUM"] / motor["INSURED_VALUE"].replace(0, np.nan)

# Policy duration
motor["INSR_BEGIN"] = pd.to_datetime(motor["INSR_BEGIN"], format="%d-%b-%y", errors="coerce")
motor["INSR_END"]   = pd.to_datetime(motor["INSR_END"],   format="%d-%b-%y", errors="coerce")
motor["policy_duration_days"] = (motor["INSR_END"] - motor["INSR_BEGIN"]).dt.days

print("Motor after cleaning:", motor.shape)
motor.head()

Motor after cleaning: (508408, 19)


,SEX,INSR_BEGIN,INSR_END,EFFECTIVE_YR,INSR_TYPE,INSURED_VALUE,PREMIUM,OBJECT_ID,PROD_YEAR,SEATS_NUM,CARRYING_CAPACITY,TYPE_VEHICLE,CCM_TON,MAKE,USAGE,CLAIM_PAID,had_claim,premium_rate,policy_duration_days
0,0,2017-08-08,2018-08-07,08,1202,519755.22,5097.83,5000029885,2007.0,4.0,6.0,Pick-up,3153.0,NISSAN,Own Goods,0.0,0,0.009808,364
1,0,2016-08-08,2017-08-07,08,1202,519755.22,6556.52,5000029885,2007.0,4.0,6.0,Pick-up,3153.0,NISSAN,Own Goods,0.0,0,0.012615,364
2,0,2015-08-08,2016-08-07,08,1202,519755.22,6556.52,5000029885,2007.0,4.0,6.0,Pick-up,3153.0,NISSAN,Own Goods,0.0,0,0.012615,365
3,0,2014-08-08,2015-08-07,08,1202,519755.22,5102.83,5000029885,2007.0,4.0,6.0,Pick-up,3153.0,NISSAN,Own Goods,0.0,0,0.009818,364
4,0,2017-08-08,2018-08-07,08,1202,1400000.00,13304.87,5000029901,2010.0,4.0,7.0,Pick-up,2494.0,TOYOTA,Own Goods,0.0,0,0.009503,364


In [12]:
claims_encoded.to_csv("../data/processed/claims_clean.csv", index=False)
motor.to_csv("../data/processed/motor_clean.csv", index=False)
print("Saved to data/processed/")

Saved to data/processed/
